# F2-vectors — Session 1: Vectors, Norms, and Distance

*One class session, roughly 75 minutes. Prerequisite: F1-scientific-python
(arrays, elementwise operations, broadcasting, axis aggregations).*

**This session:** what a vector is (and why a 1-D NumPy array already is
one), two ways to measure a vector's size — the Euclidean and Manhattan
norms — distances between data points, and how to compute norms for a whole
batch of vectors at once.

Try every checkpoint by hand first, then verify with NumPy.
Answers are collected at the end of this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Vectors as Data Points

**Motivation.**
Real measurements rarely come one at a time.
A weather station reports (temperature, humidity, wind speed).
A song can be summarized as (duration, tempo, loudness).
A point on a map is (east, north).
In every case, one *thing* is described by an ordered list of numbers —
and we want to do math on the whole list at once.

**Definition.**
A **vector** is an ordered list of numbers.
A 1-D NumPy array *is* a vector — there is nothing to convert.
The number of entries is the vector's **dimension**:
$(3, 2)$ is a 2-dimensional vector, $(215, 128, -7.5)$ is 3-dimensional.

Two mental pictures for the same vector $v = (3, 2)$:

- **a point**: start at the origin, go 3 to the right and 2 up;
- **an arrow**: an arrow from the origin to that point.

Both pictures work in 2-D and 3-D.
In 10 dimensions we can no longer draw the picture, but *every formula in
this unit still works unchanged* — that is the whole reason to learn the
algebraic version of each idea alongside its picture.

One wording warning: NumPy calls `np.array([3.0, 2.0])` a "1-D array"
(one axis), while as a *vector* it is 2-dimensional (two entries).
We will say "a vector with 2 entries" whenever it could be confusing.

**Worked example.**
Below we build two vectors and draw each one both ways:
`plt.quiver` draws arrows, and `plt.scatter` (as covered in
F1-scientific-python) marks the tips as points.

In [ ]:
# A 1-D NumPy array IS a vector.
v = np.array([3.0, 2.0])
w = np.array([-2.0, 1.0])
song = np.array([215.0, 128.0, -7.5])   # (duration in s, tempo in bpm, loudness in dB)

print("v =", v, "   entries:", v.shape[0])
print("w =", w)
print("song =", song, "   entries:", song.shape[0])

In [ ]:
plt.figure(figsize=(5, 5))
# plt.quiver draws arrows; these keyword settings make 1 data unit = 1 arrow unit.
plt.quiver([0, 0], [0, 0], [v[0], w[0]], [v[1], w[1]],
           angles="xy", scale_units="xy", scale=1, color=["C0", "C1"], width=0.008)
plt.scatter([v[0], w[0]], [v[1], w[1]], color=["C0", "C1"])
plt.text(v[0] + 0.15, v[1], "v = (3, 2)")
plt.text(w[0] - 0.6, w[1] + 0.3, "w = (-2, 1)")
plt.axhline(0, color="gray", linewidth=0.5)
plt.axvline(0, color="gray", linewidth=0.5)
plt.xlim(-4, 4)
plt.ylim(-4, 4)
plt.gca().set_aspect("equal")   # equal scales, so lengths and angles look true
plt.xlabel("first entry")
plt.ylabel("second entry")
plt.title("One vector, two pictures: arrow and point")
plt.show()

### Checkpoint 1

1. A pizza order is: 2 large pizzas, 1 salad, 4 drinks. Write it as a NumPy
   vector.
2. What is the dimension (number of entries) of `np.array([2.0, 1.0, 0.0, 7.0])`?
3. Without plotting: for the vector $(-2, 3)$, which way does the arrow point
   (left/right, up/down), and in which quadrant does its tip land?

## 2. The Euclidean Norm

**Motivation.**
Once data points are vectors, the most basic question is *how big is this
vector?*
A norm is a ruler for vectors, and the first ruler to learn is straight-line
length.

**Where the formula comes from.**
Take $v = (3, 4)$.
Its arrow is the hypotenuse of a right triangle with legs 3 and 4, so by the
Pythagorean theorem its length is $\sqrt{3^2 + 4^2} = 5$.

In 3-D, apply the Pythagorean theorem twice: for $v = (x, y, z)$, the
diagonal across the floor has length $\sqrt{x^2 + y^2}$, and that diagonal
together with the vertical leg $z$ forms another right triangle, giving
$$\sqrt{\left(\sqrt{x^2+y^2}\right)^{2} + z^2} = \sqrt{x^2 + y^2 + z^2}.$$
The same argument keeps stacking, one entry at a time, in any dimension —
which is where the general formula comes from.

**Definition (Euclidean norm).**
$$\|v\| = \sqrt{\textstyle\sum_i v_i^{\,2}}$$

**Worked example.**
By hand for $v = (3, 4)$: $\|v\| = \sqrt{9 + 16} = 5$.
And for $u = (1, -2, 2)$: $\|u\| = \sqrt{1 + 4 + 4} = 3$ — note the $-2$
contributes $+4$; squaring erases signs, so a norm is never negative.
Now with NumPy, both from the F1 pieces and with the built-in shortcut
`np.linalg.norm`:

In [ ]:
v = np.array([3.0, 4.0])
u = np.array([1.0, -2.0, 2.0])

print(np.sqrt(np.sum(v**2)), np.linalg.norm(v))   # the formula vs the shortcut
print(np.sqrt(np.sum(u**2)), np.linalg.norm(u))

### Checkpoint 2

1. By hand: compute the Euclidean norm of $(5, 12)$, and of $(2, 0, -3, 6)$.
2. True or false, with a one-line reason: for every vector $v$ and number
   $c > 0$, $\|c\,v\| = c\,\|v\|$.

## 3. The Manhattan Norm

**Motivation.**
In Manhattan you cannot walk through buildings: to get 3 blocks east and
4 blocks north you walk 7 blocks, not 5.
Straight-line length is not the only useful ruler — some situations move
along one coordinate at a time.

**Definition (Manhattan norm).**
$$\|v\|_1 = \textstyle\sum_i |v_i|$$

**When each norm fits.**
Use Euclidean when straight-line, as-the-crow-flies length is the honest
measure — physical space, and most of the geometry in this course.
Use Manhattan when movement or cost happens along one coordinate at a time
(street grids, per-entry edit costs).

The two rulers also react differently to the *shape* of a vector.
They agree exactly when a single entry carries all the size (a straight
line down one street needs no corner-cutting), and they drift apart — by up
to a factor of $\sqrt d$ in $d$ dimensions — when the size is spread evenly
across entries: the Manhattan norm credits every entry in full, while the
Euclidean norm concentrates on the largest ones.
When per-entry accumulation is the honest cost, that spread-sensitivity is
exactly what you want.

**Worked example.**
For $v = (3, 4)$: $\|v\|_1 = 3 + 4 = 7$, versus $\|v\| = 5$.
The Manhattan norm is never smaller than the Euclidean norm — the city-block
walk cannot beat the straight line.
Now the shape effect: a lone spike $(6, 0, 0, 0)$ measures 6 under *both*
rulers, while the evenly spread $(3, 3, 3, 3)$ — the *same* Euclidean
size — has Manhattan norm 12, twice as large ($\sqrt4 = 2$, the biggest gap
possible with 4 entries):

In [ ]:
v = np.array([3.0, 4.0])
print("Euclidean :", np.linalg.norm(v), "   Manhattan :", np.sum(np.abs(v)))
print("shortcut for Manhattan:", np.linalg.norm(v, 1))

spike = np.array([6.0, 0.0, 0.0, 0.0])    # all the size in one entry
spread = np.array([3.0, 3.0, 3.0, 3.0])   # same Euclidean size, spread evenly

print("spike  — Euclidean:", np.linalg.norm(spike),
      "  Manhattan:", np.sum(np.abs(spike)))    # 6 and 6: the rulers agree
print("spread — Euclidean:", np.linalg.norm(spread),
      "  Manhattan:", np.sum(np.abs(spread)))   # 6 and 12: a sqrt(4) = 2x gap

### Checkpoint 3

1. By hand: compute the Manhattan norm of $(1, -2, 2)$ and of $(5, 12)$.
2. Give a nonzero vector whose Manhattan norm *equals* its Euclidean norm,
   and say in one sentence why your example works.
3. By hand: compute both norms of $(1, 1, 1, 1)$ and the ratio
   Manhattan-over-Euclidean. How does the ratio relate to the largest gap
   possible with 4 entries?

## 4. Distance Between Points

**Motivation.**
Right behind "how big is this vector?" comes "how far apart are these two
data points?"
A distance is a norm applied to the *gap* between two points.

**Definition.**
The **Euclidean distance** between points $a$ and $b$ is $\|a - b\|$, and
the **Manhattan distance** is $\|a - b\|_1$: subtract, then measure the
difference vector with the ruler of your choice.

**Worked example.**
For $a = (1, 1)$ and $b = (4, 3)$: the difference is $a - b = (-3, -2)$, so
the Euclidean distance is $\sqrt{9 + 4} = \sqrt{13} \approx 3.61$ and the
Manhattan distance is $3 + 2 = 5$.

In [ ]:
a = np.array([1.0, 1.0])
b = np.array([4.0, 3.0])
diff = a - b

print("Euclidean distance:", np.sqrt(np.sum(diff**2)))
print("Manhattan distance:", np.sum(np.abs(diff)))

In [ ]:
plt.figure(figsize=(5.5, 4))
plt.plot([1, 4], [1, 3], "C0--", label="straight-line path (Euclidean, length 3.61)")
plt.plot([1, 4, 4], [1, 1, 3], "C1-", label="city-block path (Manhattan, length 5)")
plt.scatter([1, 4], [1, 3], color="black", zorder=3)
plt.text(0.85, 1.2, "a")
plt.text(4.08, 3.0, "b")
plt.gca().set_aspect("equal")
plt.xlabel("east")
plt.ylabel("north")
plt.title("Two rulers for the same pair of points")
plt.legend()
plt.show()

### Checkpoint 4

1. By hand: compute the Euclidean and Manhattan distances between
   $a = (2, 1)$ and $b = (5, 5)$.
2. Verify both with NumPy in one cell.

## 5. Row Norms: Many Vectors at Once

**Motivation.**
Data rarely arrives one vector at a time — it arrives as a 2-D array with
one vector *per row*.
Computing every row's norm with the F1 broadcasting-and-axis toolkit, in one
expression and with no loops, is a skill the exam tests directly.

**The recipe.**
For a 2-D array `P` of shape `(n, d)` (n vectors, d entries each):

1. `P**2` squares every entry — shape `(n, d)`;
2. `np.sum(..., axis=1)` adds *across each row* — shape `(n,)`;
3. `np.sqrt(...)` finishes — one Euclidean norm per row.

The same skeleton with `np.abs` and no root gives all the Manhattan norms.
Subtracting a query point first (broadcasting a `(d,)` array against
`(n, d)`, as covered in F1-scientific-python) turns row norms into row
*distances*.

**Worked example.**
Four points, one query, every distance at once — then `np.argmin` names the
closest point:

In [ ]:
P = np.array([[3.0, 4.0],
              [1.0, -2.0],
              [0.0, 5.0],
              [-6.0, 8.0]])

row_norms = np.sqrt(np.sum(P**2, axis=1))
print("row norms:", row_norms)                  # [5, sqrt(5), 5, 10]

q = np.array([1.0, 1.0])
diffs = P - q                                   # (4, 2) - (2,) broadcasts
dists = np.sqrt(np.sum(diffs**2, axis=1))
print("distances to q:", dists)
print("closest point is row", np.argmin(dists))

### Checkpoint 5

1. Without running code: `Z` has shape `(7, 3)`. What is the shape of
   `np.sum(Z**2, axis=1)`? And of `np.sum(Z**2, axis=0)`? Which one is
   involved in row norms?
2. By hand: compute the row norms of
   `np.array([[6.0, 8.0], [0.0, -2.0]])`.

## 6. Worked Exam-Style Example

Exam problems about norms rarely say "compute the norm" — they wrap the
answer in a **normal form** so that only one exact answer decodes correctly,
and they flag the work as reasoning, not coding.
Here is one in full, solved step by step.

---

**Problem (reasoning is required; no code).**
Let $v = (6, 4)$.
Its Euclidean norm can be written in the normal form
$$\|v\| = b\,\sqrt{a}$$
where $a$ and $b$ are positive integers and $a$ is not divisible by any
perfect square larger than 1.
Compute $a + b$.

---

**Solution.**

*Step 1 — compute the squared norm.*
$\|v\|^2 = 36 + 16 = 52$.

*Step 2 — factor out the largest perfect square.*
$52 = 4 \cdot 13$, so $\|v\| = \sqrt{52} = 2\sqrt{13}$.

*Step 3 — check the normal-form condition.*
$a = 13$ has no perfect-square factor larger than 1 — the form is fully
reduced, so it is the unique valid way to report the answer.
(Stopping at $\sqrt{52} = 1\cdot\sqrt{52}$ would give $a + b = 53$ — an
answer the grader's normal form is designed to reject.)

*Step 4 — report.*
$a + b = 13 + 2 = \boxed{15}$.

The normal-form condition is not decoration: it forces every student who is
right to submit the *same* number, which is what makes short numeric answers
gradable.

### Checkpoint 6

1. Same normal form: write $\|(9, 3)\|$ as $b\sqrt{a}$ and compute $a + b$.
2. Edge case: write $\|(2, 2, 2, 2)\|$ in the same normal form and compute
   $a + b$. (What happens when the squared norm is itself a perfect
   square?)

## 7. Common Pitfalls

**Pitfall 1 — reporting the squared norm as the norm.**
$v \cdot v$-style sums of squares appear everywhere, and it is easy to
forget the final square root.
The broken version below claims $\|(3, 4)\| = 25$:

In [ ]:
v = np.array([3.0, 4.0])

norm_broken = np.sum(v**2)            # BROKEN: this is ||v||^2, not ||v||
norm_fixed = np.sqrt(np.sum(v**2))    # fix: take the square root

print("broken:", norm_broken, "   fixed:", norm_fixed)

A quick self-check: a norm has the same *units* as the data (cm, not
cm²), and $\|(3,4)\| = 25$ should feel too big for an arrow that fits in a
5-by-5 box.
Squared norms are genuinely useful — Session 3 uses them constantly — but
they must be *labeled* as squared.

**Pitfall 2 — summing down the wrong axis.**
Row norms need `axis=1` (sum across each row's entries).
`axis=0` silently computes something else — one number per *column* — and
the only symptom is a wrong shape:

In [ ]:
P = np.array([[3.0, 4.0],
              [1.0, -2.0],
              [0.0, 5.0],
              [-6.0, 8.0]])          # 4 vectors, 2 entries each

broken = np.sqrt(np.sum(P**2, axis=0))   # BROKEN for row norms
fixed = np.sqrt(np.sum(P**2, axis=1))

print("broken:", broken, "  shape", broken.shape)   # (2,) — one value per column
print("fixed :", fixed, "  shape", fixed.shape)     # (4,) — one norm per row

The habit that catches this instantly: *predict the output shape before
running*.
Four vectors must produce four norms — if the result has shape `(2,)`, the
axis is wrong, no matter how plausible the numbers look.

### Checkpoint 7

1. A classmate computes the distance between $(1, 2)$ and $(4, 6)$ as
   `np.sum((a - b)**2)` and reports 25. What did they actually compute, and
   what is the correct distance?
2. `Q` has shape `(50, 3)`. Predict the shapes of
   `np.sqrt(np.sum(Q**2, axis=0))` and `np.sqrt(np.sum(Q**2, axis=1))`,
   and say which one is the row norms.

## Exam Connections

How this session's material shows up in Round 1 (paraphrased from the
`reference/analysis.md` topic table — no real test text here):

- The **linear-algebra cluster** — the largest block of sub-parts on the
  exam — leans on norms and distances as *ingredients*: answers are often
  requested in the $b\sqrt a$ normal form of Section 6, with the reduction
  condition doing the answer-checking.
- The **NumPy implementation cluster** poses exactly Section 5's task shape:
  batch computations over the rows of a 2-D array, with loops banned, an
  explicit shape contract, and zero points for using a banned shortcut
  function.
  Practice problems p10 and p16 train that register directly.
- Distance-versus-similarity judgment calls (which ruler fits which data
  situation) appear as short scenario sub-parts; p12 and p20 rehearse them.

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. `np.array([2.0, 1.0, 4.0])` — any fixed order of (pizzas, salads,
   drinks) works, as long as you stay consistent.
2. 4 entries, so as a vector it is 4-dimensional (NumPy still calls it a
   1-D array — one axis).
3. Left 2 and up 3; the tip lands in the upper-left (second) quadrant.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. $\sqrt{25 + 144} = 13$; and $\sqrt{4 + 0 + 9 + 36} = 7$.
2. True: $\|c\,v\| = \sqrt{\sum_i (c\,v_i)^2} = \sqrt{c^2 \sum_i v_i^2} =
   c\,\|v\|$ since $c > 0$ — scaling the vector scales its length.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. $1 + 2 + 2 = 5$; and $5 + 12 = 17$.
2. Any vector with a single nonzero entry, e.g. $(0, 7, 0)$: both norms are
   7, because with one entry there is no "corner to cut" — the straight
   line and the city-block path coincide.
3. Euclidean $\sqrt{4} = 2$, Manhattan $4$, ratio $4/2 = 2 = \sqrt{4}$: the
   perfectly even spread *achieves* the largest gap possible with 4
   entries.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. $a - b = (-3, -4)$: Euclidean $\sqrt{9 + 16} = 5$, Manhattan $3 + 4 = 7$.
2. E.g. `d = a - b; print(np.sqrt(np.sum(d**2)), np.sum(np.abs(d)))` →
   `5.0 7.0`.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. `axis=1` → shape `(7,)`; `axis=0` → shape `(3,)`. Row norms use
   `axis=1` — one number per vector.
2. $\left(\sqrt{36 + 64},\ \sqrt{0 + 4}\right) = (10,\ 2)$.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. $\|(9,3)\|^2 = 90 = 9 \cdot 10$, so $\|(9,3)\| = 3\sqrt{10}$ and
   $a + b = 13$.
2. $\|(2,2,2,2)\|^2 = 16$, a perfect square, so the norm is exactly
   $4 = 4\sqrt{1}$: $a = 1$, $b = 4$, $a + b = 5$. When the squared norm is
   a perfect square, the root disappears and $a$ collapses to 1.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. They computed the *squared* Euclidean distance $\|a-b\|^2 = 9 + 16 = 25$;
   the distance is $\sqrt{25} = 5$.
2. `axis=0` → `(3,)` (one value per column); `axis=1` → `(50,)` — the row
   norms, one per vector.

</details>

---

**Next session:** the dot product — the single number that says whether two
vectors point the same way — and cosine similarity, in
`lessons/02-dot-product-and-similarity.ipynb`.